# Zadaća 13

Preuzmite [dataset diabetes](https://www.kaggle.com/datasets/hasibur013/diabetes-dataset). Istrenirajte SVC modele za kernelima poly i rbf koji su obrađeni na 13. predavanju, ali za svaki izaberite po jedan hiperparametar i isprobajte po dvije vrijednosti. Ispišite koje koji model je najbolji, odnosno hiperparametre najboljeg modela.

In [41]:
import pandas as pd
from sklearn import svm
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [42]:
df = pd.read_csv("diabetes_dataset.csv", low_memory=False)

In [43]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [44]:
df.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Dataset ima jednu klasifikacijsku varijablu i to je `Outcome`. Ne postoje vrijednosti koje nedostaju.

In [45]:
target = df['Outcome']
features = df.drop(columns=['Outcome'])

In [46]:
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2, random_state=42)

# SVC

In [47]:
sv_classifiers = {
    "LinearSVC"    : svm.LinearSVC(),
    "SVC(linear)"  : svm.SVC(kernel='linear'),
    "SVC(poly=1)"  : svm.SVC(kernel='poly', degree=1),
    "SVC(poly=2)"  : svm.SVC(kernel='poly', degree=2),
    "SVC(poly=3)"  : svm.SVC(kernel='poly', degree=3),
    "SVC(rbf,.5)"  : svm.SVC(kernel='rbf', gamma=0.5),
    "SVC(rbf,1.0)" : svm.SVC(kernel='rbf', gamma=1.0),
    "SVC(rbf,2.0)" : svm.SVC(kernel='rbf', gamma=2.0)}

for name, clf in sv_classifiers.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    pipe.fit(features_train, target_train)
    acc = accuracy_score(target_test, pipe.predict(features_test))
    print(f"{name:12s} | accuracy = {acc:.4f}")

LinearSVC    | accuracy = 0.7532
SVC(linear)  | accuracy = 0.7597
SVC(poly=1)  | accuracy = 0.7662
SVC(poly=2)  | accuracy = 0.6948
SVC(poly=3)  | accuracy = 0.7468
SVC(rbf,.5)  | accuracy = 0.6883
SVC(rbf,1.0) | accuracy = 0.6623
SVC(rbf,2.0) | accuracy = 0.6234


Najbolji model je SVC(poly=1).

In [48]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", svm.SVC(kernel="poly", degree=1))])

param_grid = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__gamma": ["scale", "auto", 0.01, 0.1, 1, 10]}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1)

grid.fit(features_train, target_train)
predictions = grid.best_estimator_.predict(features_test)
print(f"{accuracy_score(target_test, predictions):.4f}")
print(grid.best_params_)

0.7597
{'clf__C': 0.1, 'clf__gamma': 10}
